# 8. Machine-Learning Integration

```{admonition} This chapter is not executed during the build
:class: warning
Unlike the other tutorials, this chapter is **not run** when the site is built:
it needs PyTorch and the ~500&nbsp;MB chignolin dataset (downloaded from Google
Drive), neither of which is a build requirement. The code below is the real,
working recipe — run it locally after downloading the data with
`python -m genepie.tests.download_test_data`. Expected outputs are described in
prose beneath each cell.
```

The point of genepie is that GENESIS results are ordinary NumPy arrays, so they
drop straight into the scientific-Python and deep-learning stack. This chapter
takes the folding trajectory of **chignolin** (a 10-residue mini-protein) and:

1. loads it through genepie,
2. bridges to **MDTraj** and **MDAnalysis**,
3. reduces the conformational ensemble with **scikit-learn**, and
4. trains a tiny **PyTorch** autoencoder on the coordinates.


In [ ]:
import numpy as np
from genepie import genesis_exe, SMolecule
from genepie.tests.conftest import CHIGNOLIN_PDB, CHIGNOLIN_PSF, CHIGNOLIN_DCD

# Download once with:  python -m genepie.tests.download_test_data
mol = SMolecule.from_file(pdb=CHIGNOLIN_PDB, psf=CHIGNOLIN_PSF, ref=CHIGNOLIN_PDB)
trajs, ca_mol = genesis_exe.crd_convert(
    mol, trj_files=[str(CHIGNOLIN_DCD)], trj_format="DCD", trj_type="COOR+BOX",
    selection="an:CA", fitting_selection="an:CA", fitting_method="TR+ROT",
)
traj = trajs[0]
print(traj.coords.shape)   # e.g. (10000, 10, 3): thousands of frames, 10 CA atoms

*Expected:* a `(n_frames, 10, 3)` array — a few thousand fitted C&alpha; frames.

In [ ]:
# RMSD to the folded reference: a natural 1-D reaction coordinate.
rmsd = genesis_exe.rmsd_analysis(
    mol, traj, analysis_selection="an:CA",
    fitting_selection="an:CA", fitting_method="TR+ROT",
).rmsd
print(rmsd.min(), rmsd.max())   # spans folded (~1 A) to unfolded (several A)

## Bridge to MDTraj / MDAnalysis

In [ ]:
# SMolecule/STrajectories convert to both toolkits, so their analyses and writers
# are available on the same data.
top = mol.to_mdtraj_topology()
universe = mol.to_mdanalysis_universe()
print(top.n_atoms, universe.atoms.n_atoms)

## Dimensionality reduction with scikit-learn

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

X = traj.coords.reshape(traj.nframe, -1)          # (n_frames, 3*n_ca)
pcs = PCA(n_components=2).fit_transform(X)         # 2-D conformational map
states = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(pcs)
# `pcs` is a 2-D embedding; `states` labels metastable basins (folded / intermediate / unfolded).

*Expected:* a 2-D PCA embedding whose density separates into folded and unfolded basins, with three k-means labels.

In [ ]:
# Plot the free-energy-like landscape (negative log density) coloured by state.
import numpy as np
import plotly.express as px
fig = px.scatter(x=pcs[:, 0], y=pcs[:, 1], color=states.astype(str),
                 labels={"x": "PC1", "y": "PC2", "color": "state"},
                 title="Chignolin conformational map (PCA + k-means)")
fig

## A tiny PyTorch autoencoder

In [ ]:
import torch
import torch.nn as nn

data = torch.tensor(X, dtype=torch.float32)
n_in = data.shape[1]

class AE(nn.Module):
    def __init__(self, n_in, latent=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(n_in, 32), nn.ReLU(), nn.Linear(32, latent))
        self.dec = nn.Sequential(nn.Linear(latent, 32), nn.ReLU(), nn.Linear(32, n_in))
    def forward(self, x):
        return self.dec(self.enc(x))

model = AE(n_in)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
for epoch in range(200):
    opt.zero_grad()
    loss = loss_fn(model(data), data)
    loss.backward()
    opt.step()
print("final reconstruction loss:", float(loss))

# The 2-D latent space (model.enc(data)) is another learned reaction coordinate,
# directly comparable to the PCA map above.

*Expected:* the reconstruction loss decreases steadily over the 200 epochs, and
the 2-D latent space separates folded from unfolded conformations — a learned
reaction coordinate obtained end-to-end from GENESIS output without leaving Python.

This closes the loop the introduction promised: **setup → simulation → analysis →
ML**, all as one programmable NumPy-native workflow.
